# Site Load Quantile Forecast Sanity Check (Issue #11)

This notebook inspects the LightGBM quantile regression forecaster ($P_{10}/P_{50}/P_{90}$) for site inflexible base load and compares performance against the 24-hour persistence baseline.

In [ ]:
import datetime
import json
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import pandas as pd

from energy_brain.forecast.load import predict

# Load metrics
with open("../reports/forecast_metrics.json", encoding="utf-8") as f:
    metrics = json.load(f)["load"]

print("=== Load Forecast Metrics ===")
print(f"Persistence Baseline MAE: {metrics['persistence_mae_kw']:.2f} kW")
print(f"LightGBM P50 MAE:         {metrics['lightgbm_p50_mae_kw']:.2f} kW")
print(f"MAE Improvement:          +{metrics['mae_improvement_pct']:.1f}%")
print(f"P10-P90 Coverage:         {metrics['p10_p90_coverage_pct']:.1f}% (Nominal: 80%)")

## 24-Hour Day-Ahead Load Forecast vs Actuals vs Persistence

In [ ]:
DHAKA = ZoneInfo("Asia/Dhaka")
test_dt = datetime.datetime(2024, 6, 3, tzinfo=DHAKA)

pred_df = predict(test_dt).set_index("timestamp")
load_df = pd.read_parquet("../data/processed/load.parquet").set_index("timestamp")
actual = load_df.loc["2024-06-03"]["base_kw"]
pers = load_df.loc["2024-06-02"]["base_kw"].values

plt.figure(figsize=(12, 5))
plt.plot(
    pred_df.index.strftime("%H:%M"),
    actual.values,
    label="Actual Base Load (June 3)",
    color="black",
    lw=2,
)
plt.plot(
    pred_df.index.strftime("%H:%M"),
    pers,
    label="Persistence Baseline (June 2)",
    color="gray",
    linestyle="--",
    lw=1.5,
)
plt.plot(
    pred_df.index.strftime("%H:%M"),
    pred_df["p50_kw"],
    label="LightGBM P50 (Forecast)",
    color="tab:blue",
    lw=2,
)
plt.fill_between(
    pred_df.index.strftime("%H:%M"),
    pred_df["p10_kw"],
    pred_df["p90_kw"],
    color="lightskyblue",
    alpha=0.4,
    label="P10 - P90 Uncertainty Band",
)
plt.xticks(pred_df.index.strftime("%H:%M")[::8], rotation=45)
plt.title("24-Hour Base Load Quantile Forecast (Ashulia Factory - June 3, 2024)")
plt.xlabel("Time of Day (Asia/Dhaka)")
plt.ylabel("Inflexible Base Load (kW)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()